In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
print(sys.platform)
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


linux


## Local MCP server

In [15]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [16]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [17]:
# from langchain.agents import create_agent
# from langchain_google_genai import ChatGoogleGenerativeAI
# from dotenv import load_dotenv
# import os
# load_dotenv()

# model = ChatGoogleGenerativeAI(
#     model="gemini-3-flash-preview",
#     google_api_key=os.getenv("GOOGLE_API_KEY"),
# )

# agent = create_agent(
#     model=model,
#     tools=tools,
#     system_prompt=prompt
# )

In [18]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os
load_dotenv()

ollama_host_server=os.environ["OLLAMA_HOST_SERVER"]

model = ChatOllama(model="qwen3:1.7b", base_url=ollama_host_server)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [20]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='b1791645-3bdf-4744-b1c9-20b618df3c9a'),
              AIMessage(content="I'm sorry, I can only answer questions about LangChain, LangGraph and LangSmith. \n\nHowever, I can help you search for information about the langchain-mcp-adapters library using the web. Would you like me to perform a search for this library?", additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-10-24T19:30:20.603684196Z', 'done': True, 'done_reason': 'stop', 'total_duration': 67360632901, 'load_duration': 807328, 'prompt_eval_count': 276, 'prompt_eval_duration': 151606000, 'eval_count': 411, 'eval_duration': 67156931000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a16f63-7301-79a2-bc47-98812b81d20a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 411, 'total

## Online MCP

In [26]:

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [27]:
agent = create_agent(
    model=model,
    tools=tools
)

In [28]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='de93da13-7f50-4c93-a50d-e2a166e63743'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-10-24T19:33:36.779920999Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24900305986, 'load_duration': 2055719, 'prompt_eval_count': 337, 'prompt_eval_duration': 156105000, 'eval_count': 157, 'eval_duration': 24685313000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a16f67-1a15-71f1-8014-25fea162a2cd-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': '35753b77-55d2-4778-98b2-83c2011bc488', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 337, 'output_tokens': 157, 'total_tokens': 494}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-1

## Trip MCP

In [32]:

client = MultiServerMCPClient(
    {
        "instavision": {
          "transport": "stdio",
          "command": "npx",
          "args": [
            "-y",
            "instavision-mcp"
          ],
          "env": {
            "INSTAVISION_API_KEY": "iv_sk__2-l1MFBkLpPt-sYEmDrXPAHGGaiW0Ned6asQq36Okw"
          }
        }
    }
)

tools = await client.get_tools()

In [33]:
agent = create_agent(
    model=model,
    tools=tools
)

In [34]:
from langchain.messages import HumanMessage
from pprint import pprint

question = HumanMessage(content="Find fitness coaches in New York with 5k to 50k followers and a public email. Just list 2 coaches")

config = {"configurable": {"thread_id": "4"}}

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

{'messages': [HumanMessage(content='Find fitness coaches in New York with 5k to 50k followers and a public email. Just list 2 coaches', additional_kwargs={}, response_metadata={}, id='19cc9a64-b6c3-408b-bbb1-0f3f78b37166'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-10-24T19:45:40.704929709Z', 'done': True, 'done_reason': 'stop', 'total_duration': 109663031742, 'load_duration': 2456328, 'prompt_eval_count': 1861, 'prompt_eval_duration': 2043010000, 'eval_count': 402, 'eval_duration': 107602275000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a16f70-d875-7303-85b7-5bf196f9c048-0', tool_calls=[{'name': 'launch_discovery', 'args': {'activeWithinDays': 'any', 'creditsCap': 500, 'excludeSeen': True, 'followerSize': 'mid', 'location': 'New York', 'locationMode': 'only', 'request': 'fitness coaches in New York with 5k to 50k followers and a public email', 'accountKind': 'cr

In [37]:

client = MultiServerMCPClient(
    {
        "citysignal-ai": {
          "transport": "http",
          "url": "https://citysignalai.com/mcp"
        }
    }
)

tools = await client.get_tools()

agent = create_agent(
    model=model,
    tools=tools
)

In [38]:
from langchain.messages import HumanMessage
from pprint import pprint

question = HumanMessage(content="What can you do?")

config = {"configurable": {"thread_id": "3"}}

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

{'messages': [HumanMessage(content='What can you do?', additional_kwargs={}, response_metadata={}, id='d0c5b11c-4f5c-42bd-bdd6-173a67dbc7d4'),
              AIMessage(content="The tools I can perform include managing your CitySignal account, tracking your day's alerts, briefings, routes, errands, spots, and circle interactions. Key functions:  \n1. **Account Management**: Verify your profile, links, and permissions.  \n2. **Daily Insights**: Get today's alerts, calendar lines, and summaries (my_day/my_brief).  \n3. **Route & Errand Tracking**: Monitor delays, detours, and nearby errands (my_routes_now/my_along_the_way).  \n4. **Shopping Lists**: Manage groceries, sales, and shared items (my_list).  \n5. **Spots & Routes**: Locate saved places/routes (my_spots_and_routes).  \n6. **Customized Alerts**: Receive alerts for events, roadwork, and weather (my_day).  \n7. **Circle Interactions**: Share needs/errands with people in your Circle (circle_needs/circle_along_the_way).  \n\n**Require